# Week 10 (evaluation): conditional sampling and distributional verification

This notebook is the evaluation half of Week 10. It assumes `10a_conditioned_train.ipynb` has been run end-to-end and has produced `ckpt_conditional.ckpt` in this directory. It *also* assumes Week 08 has produced `ckpt_full.ckpt` (the unconditional t-aware model) — the unconditional model is the natural baseline against which we measure whether the conditioning machinery actually does anything useful at the distributional level.

The headline value-add metric — `compute_global_nll` evaluated on the held-out validation split, classical alone vs. classical + conditional residuals — lives in its own notebook. *This* notebook answers the prior question: does the conditional model produce samples that are sensitive to the conditioning in the way we expect, and do those samples look like training residuals at the distributional level?

The Week 10 evaluation tasks pick up where Week 08's Task 44 left off:

- **Load** both checkpoints and re-verify the t-sensitivity *and* cond-sensitivity of the loaded conditional model (the analogue of Week 08's "redo Task 39 on the loaded model" defence).

- **Task 53**: sample residuals from the conditional model with a small set of distinct conditioning vectors, visualize how the samples qualitatively shift as `cond` varies.

- **Task 54**: distributional verification — for each validation window, sample N residuals at that window's conditioning, then compare the aggregate sampled distribution against the held-out validation residuals. The unconditional model serves as the reference baseline: does the conditional model do *better* at matching the validation distribution, or has the conditioning machinery learned nothing useful at this scale?


In [ ]:
import os, subprocess, sys

# Keep this path if working in Colab
# repo_path = "/content/butterflai"

# Use this path if working locally
repo_path = "../../"

if not os.path.isdir(repo_path):
    subprocess.run(["git", "clone", "https://github.com/SwRI-IDEA-Lab/butterflai.git", repo_path], check=True)
else:
    try:
        subprocess.run(["git", "-C", repo_path, "pull"], check=True)
    except subprocess.CalledProcessError as e:
        print(f"git pull skipped: {e}")
sys.path.insert(0, repo_path)
from infrastructure.utils.colab_setup import setup
setup()


In [ ]:
%load_ext autoreload
%autoreload 2

import os, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

import torch
from einops import repeat


In [ ]:
# ── locate Week 08/09/10 artifacts (split across three folders) ────────────
def _find(filename, search_dirs):
    for d in search_dirs:
        p = os.path.join(d, filename)
        if os.path.isfile(p):
            return p
    return None

_cwd = os.getcwd()
_search_dirs = []
# Always include the current directory first — ckpt_conditional.ckpt was
# saved by 10a into this directory, so the local path should win.
if os.path.isdir(_cwd):
    _search_dirs.append(_cwd)
for _base in [_cwd] + [os.path.abspath(os.path.join(_cwd, *[".."] * i)) for i in range(0, 5)]:
    for _sub in [("weeks", "week_10"), ("weeks", "week_09"), ("weeks", "week_08")]:
        _candidate = os.path.join(_base, *_sub)
        if os.path.isdir(_candidate) and _candidate not in _search_dirs:
            _search_dirs.append(_candidate)
    if (any(w in _base for w in ("week_08", "week_09", "week_10"))
            and os.path.isdir(_base) and _base not in _search_dirs):
        _search_dirs.append(_base)

_unconditioned_py  = _find("unconditioned_infrastructure.py", _search_dirs)
_conditioned_py    = _find("conditioned_infrastructure.py",   _search_dirs)
_parquet_path      = _find("diffusion_windows.parquet", _search_dirs)
_classical_py      = _find("butterflAI_model.py",     _search_dirs)
_classical_weights = _find("official_model.npz",      _search_dirs)
_ckpt_uncond_path  = _find("ckpt_full.ckpt",          _search_dirs)
_ckpt_cond_path    = _find("ckpt_conditional.ckpt",   _search_dirs)

_missing = [n for n, p in [
    ("unconditioned_infrastructure.py", _unconditioned_py),
    ("conditioned_infrastructure.py",   _conditioned_py),
    ("diffusion_windows.parquet", _parquet_path),
    ("butterflAI_model.py",     _classical_py),
    ("official_model.npz",      _classical_weights),
    ("ckpt_full.ckpt",          _ckpt_uncond_path),
    ("ckpt_conditional.ckpt",   _ckpt_cond_path),
] if p is None]
if _missing:
    raise FileNotFoundError(
        f"Cannot locate {_missing} under weeks/week_08, weeks/week_09, or weeks/week_10. "
        f"Searched: {_search_dirs}"
    )

_repo_root = os.path.abspath(os.path.join(os.path.dirname(_conditioned_py), "..", ".."))
for _p in [_repo_root,
           os.path.dirname(_unconditioned_py),
           os.path.dirname(_conditioned_py),
           os.path.dirname(_classical_py)]:
    if _p not in sys.path:
        sys.path.insert(0, _p)

# Week 08 reused machinery + Week 09 conditional classes/functions
from unconditioned_infrastructure import (
    make_cosine_schedule, ResidualDataset,
    DiffusionMLP, DiffusionLightning, sample,
)
from conditioned_infrastructure import (
    ConditionalResidualDataset,
    ConditionalDiffusionMLP,
    ConditionalDiffusionLightning,
    sample_conditional,
)

# ── data (same parquet as training) ────────────────────────────────────────
windows_df = pd.read_parquet(_parquet_path)

LAT_BINS    = np.linspace(0, 45, 16)
BIN_WIDTH   = 3.0
BIN_CENTERS = 0.5 * (LAT_BINS[:-1] + LAT_BINS[1:])

# Schedule arrays — used as placeholders at load time; the actual values
# come from the checkpoint's saved buffers.
T = 200
alpha_np, sigma_np, _ = make_cosine_schedule(T=T, s=0.008)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
print(f"Loaded {len(windows_df)} windows. Split sizes: "
      f"{windows_df['split'].value_counts().sort_index().to_dict()}")
print(f"ckpt_full        = {_ckpt_uncond_path}")
print(f"ckpt_conditional = {_ckpt_cond_path}")


---
## Load both checkpoints (Week 08 unconditional + Week 10 conditional)

The `load_from_checkpoint` pattern is the same as Week 08, with one addition for the conditional module: `cond_means` and `cond_stds` are now also among the buffers being restored, so the helper passes placeholder zero/one tensors that the checkpoint overwrites.

After loading, both `lightning_uncond.bin_means` and `lightning_cond.cond_means` should be **non-placeholder** values — that is, *not* all zeros. The print statements at the end of the cell let you verify this at a glance; if either looks all-zero or all-one, the corresponding statistics were not saved with the checkpoint and every sampler call downstream will silently de-normalize wrong.


In [ ]:
# ── load both checkpoints ──────────────────────────────────────────────────
def _load_unconditional(ckpt_path):
    inner = DiffusionMLP(use_timestep_embedding=True)
    lm = DiffusionLightning.load_from_checkpoint(
        ckpt_path,
        model=inner,
        alpha=alpha_np, sigma=sigma_np,
        bin_means=np.zeros(15, dtype=np.float32),
        bin_stds=np.ones(15, dtype=np.float32),
        map_location=device,
    )
    return lm.to(device).eval()

def _load_conditional(ckpt_path):
    inner = ConditionalDiffusionMLP()
    lm = ConditionalDiffusionLightning.load_from_checkpoint(
        ckpt_path,
        model=inner,
        alpha=alpha_np, sigma=sigma_np,
        bin_means=np.zeros(15, dtype=np.float32),
        bin_stds=np.ones(15, dtype=np.float32),
        cond_means=np.zeros(4, dtype=np.float32),
        cond_stds=np.ones(4, dtype=np.float32),
        map_location=device,
    )
    return lm.to(device).eval()

lightning_uncond = _load_unconditional(_ckpt_uncond_path)
lightning_cond   = _load_conditional(_ckpt_cond_path)

print("Loaded both modules.")
print(f"  uncond bin_means[:3]  = {lightning_uncond.bin_means[:3].cpu().numpy()}  "
      f"(should NOT be all zeros)")
print(f"  cond   bin_means[:3]  = {lightning_cond.bin_means[:3].cpu().numpy()}")
print(f"  cond   cond_means     = {lightning_cond.cond_means.cpu().numpy()}  "
      f"(should NOT be all zeros)")
print(f"  cond   cond_stds      = {lightning_cond.cond_stds.cpu().numpy()}  "
      f"(should NOT be all ones)")


---
## Task 49 (loaded) — re-verify t-sensitivity and cond-sensitivity on the loaded model

The training notebook ran both sanity checks on a fresh model. This is the same pair of checks on the *loaded* conditional model. If something went wrong during serialization or loading (the conditioning concatenation got disconnected, the timestep embedding's state did not transfer correctly, the cond statistics buffers are wrong), one of these will catch it.


In [ ]:
# Task 49 (loaded) — sanity checks on the conditional model after load.

torch.manual_seed(1)

COND_NAMES = ["area_smoothed", "mu_universal", "model_sigma", "amplitude"]
N_COND_DIM = len(COND_NAMES)

r_t_fixed = torch.randn(15, device=device)
t_values  = torch.tensor([0, T//4, T//2, 3*T//4, T-1], dtype=torch.long, device=device)
cond_zero = torch.zeros(N_COND_DIM, device=device)

# t-sensitivity (hold r_t, cond fixed, vary t)
r_t_batch_t  = repeat(r_t_fixed, "d -> n d", n=5)
cond_batch_t = repeat(cond_zero, "d -> n d", n=5)
with torch.no_grad():
    out_t = lightning_cond.model(r_t_batch_t, t_values, cond_batch_t)
dist_t   = (out_t[None] - out_t[:, None]).norm(dim=-1)
max_off_t = dist_t[~torch.eye(5, dtype=torch.bool, device=device)].max().item()

# cond-sensitivity: perturb each cond channel independently from a zero
# baseline. Row 0 = baseline; rows 1..N_COND_DIM each perturb one channel
# by +2 in normalized space. EACH per-channel L2 must be > 0 — that is
# the analogue of the per-channel check at the top of 10a, redone here
# on the loaded state.
cond_values = torch.zeros(1 + N_COND_DIM, N_COND_DIM, device=device)
for j in range(N_COND_DIM):
    cond_values[1 + j, j] = 2.0
n_c         = cond_values.shape[0]
r_t_batch_c = repeat(r_t_fixed, "d -> n d", n=n_c)
t_fixed     = torch.full((n_c,), T // 2, dtype=torch.long, device=device)
with torch.no_grad():
    out_c = lightning_cond.model(r_t_batch_c, t_fixed, cond_values)
dist_c         = (out_c[None] - out_c[:, None]).norm(dim=-1)
per_channel_l2 = dist_c[0, 1:]
min_off_c      = per_channel_l2.min().item()

print(f"loaded model t-sensitivity            : max off-diagonal L2 = {max_off_t:.4f}")
print(f"loaded model cond-sensitivity per chan:")
for name, l2 in zip(COND_NAMES, per_channel_l2.cpu().tolist()):
    print(f"    {name:14s}: L2 from baseline = {l2:.4f}  (must be > 0)")
print(f"loaded model cond-sensitivity worst   : min per-channel L2 = {min_off_c:.4f}")
assert max_off_t > 0, "loaded t-sensitivity is zero — timestep embedding state did not transfer"
assert min_off_c > 0, ("loaded cond-sensitivity is zero on at least one channel — "
                       "a cond input is being silently dropped after load")

# Heatmap pair for visual confirmation.
cond_labels = ["zero"] + [f"+{n}" for n in COND_NAMES]
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
im0 = axes[0].imshow(dist_t.cpu().numpy(), cmap="viridis")
axes[0].set_xticks(range(5)); axes[0].set_xticklabels([f"t={int(v)}" for v in t_values.cpu()],
                                                      rotation=30, fontsize=8)
axes[0].set_yticks(range(5)); axes[0].set_yticklabels([f"t={int(v)}" for v in t_values.cpu()], fontsize=8)
axes[0].set_title("loaded: t-sensitivity", fontsize=10)
fig.colorbar(im0, ax=axes[0], fraction=0.046, label="pairwise L2")

im1 = axes[1].imshow(dist_c.cpu().numpy(), cmap="viridis")
axes[1].set_xticks(range(n_c)); axes[1].set_xticklabels(cond_labels, rotation=30, fontsize=8)
axes[1].set_yticks(range(n_c)); axes[1].set_yticklabels(cond_labels, fontsize=8)
axes[1].set_title("loaded: cond-sensitivity (per channel)", fontsize=10)
fig.colorbar(im1, ax=axes[1], fraction=0.046, label="pairwise L2")
plt.tight_layout(); plt.show()


---
## Task 53 — Conditional sample visualization

Pick a small set of conditioning vectors spanning interesting points in `(area_smoothed, mu_universal, model_sigma, amplitude)` space, generate samples conditioned on each, and plot the results side-by-side. This is the qualitative payoff of training a conditional model: the generated residuals should look visibly different across the cond values, in ways that make some kind of physical sense.

The cleanest choice for "interesting points": three or four real validation windows from `windows_df` covering distinct phases of distinct cycles — say, an early-phase window from a tall cycle, a mid-phase window from a tall cycle, an early-phase window from a short cycle, etc. Reading the conditioning straight out of `windows_df` (then normalizing through `lightning_cond.cond_means` / `cond_stds`) means you are sampling at exactly the operating points the model will be asked to handle in the NLL evaluation, which is more meaningful than synthetic conditioning.

For each chosen validation window, generate `N_PER_COND = 50` samples and plot their mean and ±1σ band as a function of bin latitude. Overlay the *actual* held-out residual for that window in a contrasting colour. Each panel's title shows the four cond values (`area`, `μ`, `σ`, `A`) so the eye can connect each chosen window to its conditioning. The sampled distribution does not need to be perfectly centered on the actual residual — the residual is one draw from a noisy process — but the actual residual should generally lie inside or near the sampled band.


In [ ]:
# Task 53 — sample with realistic validation-window conditioning, plot mean ± σ.

N_PER_COND = 50

HIST_COLS = [f"hist_emp_{j:02d}" for j in range(15)]
PAR_COLS  = [f"hist_par_{j:02d}" for j in range(15)]
COND_COLS = ["area_smoothed", "mu_universal", "model_sigma", "amplitude"]

# Pick 4 validation windows spanning distinct (cycle, hemisphere, τ-phase) combos.
val_df = windows_df[windows_df["split"] == "val"].reset_index(drop=True)
chosen_idx = [0, len(val_df) // 4, len(val_df) // 2, 3 * len(val_df) // 4]
chosen = val_df.iloc[chosen_idx].reset_index(drop=True)

# Normalize the chosen 4-vector cond into cond-space using the *loaded*
# module's saved statistics — these are the train-split stats.
cond_raw  = torch.tensor(chosen[COND_COLS].to_numpy(np.float32), device=device)
cond_norm = (cond_raw - lightning_cond.cond_means) / lightning_cond.cond_stds

# Repeat each cond row N_PER_COND times for K independent samples per window.
cond_batched = repeat(cond_norm, "n d -> (n k) d", k=N_PER_COND)
torch.manual_seed(0)
samples = sample_conditional(lightning_cond, cond_batched, device=device).cpu().numpy()
samples = samples.reshape(len(chosen), N_PER_COND, 15)

emp = chosen[HIST_COLS].to_numpy(np.float32)
par = chosen[PAR_COLS].to_numpy(np.float32)
true_residuals = emp - par                                            # (4, 15)

fig, axes = plt.subplots(1, 4, figsize=(18, 4.5), sharey=True)
for ax, idx in zip(axes, range(len(chosen))):
    mu_s, sd_s = samples[idx].mean(0), samples[idx].std(0)
    ax.fill_between(BIN_CENTERS, mu_s - sd_s, mu_s + sd_s, alpha=0.3, color="C2",
                    label="sample ±σ")
    ax.plot(BIN_CENTERS, mu_s,                color="C2", lw=2, label="sample mean")
    ax.plot(BIN_CENTERS, true_residuals[idx], color="C0", lw=2, label="true residual")
    ax.axhline(0, color="k", linewidth=0.4)
    row = chosen.iloc[idx]
    ax.set_title(
        f"cyc {int(row['cycle'])} {row['hemisphere']}  τ={row['tau_center']:.2f}\n"
        f"area={row['area_smoothed']:.0f}  μ={row['mu_universal']:.1f}\n"
        f"σ={row['model_sigma']:.2f}  A={row['amplitude']:.0f}",
        fontsize=9,
    )
    ax.set_xlabel("|latitude| (°)")
    ax.legend(fontsize=8)
axes[0].set_ylabel("residual")
fig.suptitle(f"Conditional samples (mean ± σ over {N_PER_COND} draws) vs. held-out residual")
plt.tight_layout(); plt.show()


---
## Task 54 — Conditional distributional verification

Headline of the evaluation notebook. For each validation window, sample `N_PER_WIN` conditional residuals at that window's conditioning, then aggregate over all validation windows to form a "conditional sample distribution" comparable to the actual held-out validation residual distribution. The unconditional Week 08 model is the reference baseline: sample `N_TOTAL_UNCOND ≈ N_PER_WIN × N_VAL_WINDOWS` unconditional residuals (with no targeting at all) and treat that as the "what if we ignored conditioning" comparator.

The diagnostics from Week 08's Task 43 — bin-wise mean, bin-wise std, bin-bin covariance heatmap — applied three ways: training residuals, conditional samples (per-window targeting), unconditional samples (random). The expected reading:

- *If conditioning is working*, the conditional samples' bin-wise std should be **smaller** than the unconditional samples' (because each window's conditional distribution is narrower than the marginal), and the conditional samples' aggregate covariance should match the validation residuals' aggregate covariance *better* than the unconditional samples' does. The improvement may be small at this dataset scale — the NLL notebook will quantify it.

- *If conditioning is not working*, the conditional and unconditional comparisons will look nearly identical. That's the same failure-mode signature the cond-sensitivity check at the top of this notebook is designed to catch *before* you spend time here.

- *If conditioning is working but the trained model is overconfident*, the conditional std will be visibly *narrower* than the validation residuals' actual per-window variability — the model produces residuals that look like noise-free predictions when in fact every real window has a genuine spread of residuals around the conditional mean. This is real and worth flagging if you see it.


In [ ]:
# Task 54 — distributional comparison: training, val, conditional, unconditional.

N_PER_WIN = 20

HIST_COLS = [f"hist_emp_{j:02d}" for j in range(15)]
PAR_COLS  = [f"hist_par_{j:02d}" for j in range(15)]
COND_COLS = ["area_smoothed", "mu_universal", "model_sigma", "amplitude"]

val_df   = windows_df[windows_df["split"] == "val"].reset_index(drop=True)
train_df = windows_df[windows_df["split"] == "train"].reset_index(drop=True)
N_TOTAL  = len(val_df) * N_PER_WIN

# 1. Training residuals (physical units) for the reference.
emp = train_df[HIST_COLS].to_numpy(np.float32)
par = train_df[PAR_COLS].to_numpy(np.float32)
train_residuals_phys = emp - par
rng = np.random.default_rng(123)
train_arr = train_residuals_phys[rng.integers(0, len(train_residuals_phys), size=N_TOTAL)]

# 2. Held-out validation residuals (physical units).
emp_v = val_df[HIST_COLS].to_numpy(np.float32)
par_v = val_df[PAR_COLS].to_numpy(np.float32)
val_residuals_phys = emp_v - par_v                                   # (N_val, 15)

# 3. Conditional samples: per-window targeting on the full 4-vector cond.
cond_raw  = torch.tensor(val_df[COND_COLS].to_numpy(np.float32), device=device)
cond_norm = (cond_raw - lightning_cond.cond_means) / lightning_cond.cond_stds
cond_batched = repeat(cond_norm, "n d -> (n k) d", k=N_PER_WIN)
torch.manual_seed(0)
samples_cond = sample_conditional(lightning_cond, cond_batched, device=device).cpu().numpy()

# 4. Unconditional samples: N_TOTAL random draws (no targeting).
torch.manual_seed(0)
samples_uncond = sample(lightning_uncond, batch_size=N_TOTAL, data_dim=15,
                        device=device).cpu().numpy()

# 5. Bin-wise statistics
def _binwise(arr):
    return arr.mean(0), arr.std(0), np.cov(arr, rowvar=False)

m_train, s_train, c_train = _binwise(train_arr)
m_val,   s_val,   c_val   = _binwise(val_residuals_phys)
m_cond,  s_cond,  c_cond  = _binwise(samples_cond)
m_unc,   s_unc,   c_unc   = _binwise(samples_uncond)

# Mean / std bar panels.
fig, axes = plt.subplots(1, 2, figsize=(14, 4), sharex=True)
w = BIN_WIDTH * 0.22
for ax, mt, mh, mc, mu, title in [
    (axes[0], m_train, m_val, m_cond, m_unc, "Bin-wise mean"),
    (axes[1], s_train, s_val, s_cond, s_unc, "Bin-wise std"),
]:
    ax.bar(BIN_CENTERS - 1.5 * w, mt, width=w, label="training",       color="C0", edgecolor="black", linewidth=0.3)
    ax.bar(BIN_CENTERS - 0.5 * w, mh, width=w, label="val (held-out)", color="C4", edgecolor="black", linewidth=0.3)
    ax.bar(BIN_CENTERS + 0.5 * w, mc, width=w, label="conditional",    color="C2", edgecolor="black", linewidth=0.3)
    ax.bar(BIN_CENTERS + 1.5 * w, mu, width=w, label="unconditional",  color="C3", edgecolor="black", linewidth=0.3)
    ax.set_title(title); ax.set_xlabel("|latitude| (°)"); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

# Covariance heatmaps.
vmax = max(np.abs(c_train).max(), np.abs(c_val).max(),
           np.abs(c_cond).max(),  np.abs(c_unc).max())
fig, axes = plt.subplots(1, 4, figsize=(18, 4.5))
for ax, mat, title in [
    (axes[0], c_train, "Training"),
    (axes[1], c_val,   "Val (held-out)"),
    (axes[2], c_cond,  "Conditional samples"),
    (axes[3], c_unc,   "Unconditional samples"),
]:
    im = ax.imshow(mat, cmap="RdBu_r", vmin=-vmax, vmax=vmax)
    ax.set_title(title); ax.set_xlabel("bin"); ax.set_ylabel("bin")
    fig.colorbar(im, ax=ax, fraction=0.046)
fig.suptitle("Bin-bin covariance: training vs. val vs. conditional vs. unconditional")
plt.tight_layout(); plt.show()

# Summary table.
comparison = pd.DataFrame([
    {"row": "val (held-out)",
     "binwise_mean_MSE_vs_val": 0.0,
     "binwise_std_MSE_vs_val":  0.0,
     "cov_frob_vs_val":         0.0},
    {"row": "conditional samples",
     "binwise_mean_MSE_vs_val": float(np.mean((m_cond - m_val) ** 2)),
     "binwise_std_MSE_vs_val":  float(np.mean((s_cond - s_val) ** 2)),
     "cov_frob_vs_val":         float(np.linalg.norm(c_cond - c_val))},
    {"row": "unconditional samples",
     "binwise_mean_MSE_vs_val": float(np.mean((m_unc - m_val) ** 2)),
     "binwise_std_MSE_vs_val":  float(np.mean((s_unc - s_val) ** 2)),
     "cov_frob_vs_val":         float(np.linalg.norm(c_unc - c_val))},
])
print(comparison.to_string(index=False))


---
## Where Week 10 leaves us, and what the NLL notebook will measure

By the end of this notebook you have:

- A trained **conditional** diffusion model whose samples are visibly sensitive to the conditioning (Task 53) and that aggregates, at the distributional level, to something closer to held-out validation residuals than the unconditional baseline does (Task 54). The improvement may be small at this scale; the qualitative direction is what matters before quantification.

- A loaded, sanity-checked conditional checkpoint that the NLL notebook will use to compute the headline value-add metric.

What you have **not** done is the headline measurement: does combining the classical ButterflAI density with conditional diffusion residuals actually outperform the classical density alone, on `compute_global_nll` applied to the held-out validation split? That comparison is its own clean notebook, picking up where this one ends.

**What the NLL notebook will do.** For each validation window: read its `(area_smoothed, mu_universal, model_sigma, amplitude)` 4-vector, build the classical density on `BIN_CENTERS`, draw N conditional residual samples at that conditioning, add the samples to the classical density (with appropriate non-negativity / normalization handling), compute the per-window NLL of the actual emp histogram under that mixture. Aggregate via `compute_global_nll` across all validation windows. Compare against (a) classical alone and (b) classical + Week 08 unconditional samples to isolate how much the conditioning is worth.

That's the end-of-program payoff measurement. Everything in Week 10 was infrastructure pointed at making that measurement possible.
